# Contour Plots: CW vs 1-fs Pulsed Beam Intensity Distributions (Figure 1)

This notebook reproduces Figure 1 from Romallosa et al. (2003), showing contour plots of time-integrated focused intensity distributions near the geometrical focus.

**Parameters:**
- Wavelength: λc = 750 nm
- Numerical Aperture: XNA = 0.8  
- Refractive index: n = 1.3
- Pulse width: τ = 1 fs (for pulsed beam)

**Comparison:**
- (a) CW beam intensity distribution
- (b) 1-fs pulsed beam intensity distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add the src directory to path
sys.path.append('../src')

from optical_diffraction import OpticalSystem, PulsedBeam
from optical_diffraction import (richards_wolf_vector_diffraction, 
                                pulsed_richards_wolf_diffraction,
                                create_coordinate_grids)

# Set up plotting parameters
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

In [ ]:
# System parameters (matching Romallosa et al. 2003, Figure 1)
wavelength = 750e-9  # 750 nm center wavelength
NA = 0.8             # Numerical aperture XNA = 0.8
n = 1.3              # Refractive index
tau = 1e-15          # 1 fs pulse width

# Create optical system
system = OpticalSystem(wavelength, NA, n, "750nm, NA=0.8, n=1.3")

# Create pulsed beam
pulsed_beam = PulsedBeam(wavelength, tau)

print(f"Optical System:")
print(f"  Wavelength: {wavelength*1e9:.0f} nm")
print(f"  NA: {NA}")
print(f"  Refractive index: {n}")
print(f"  Angular aperture: {np.degrees(system.angular_aperture):.1f}°")
print(f"  Rayleigh length: {system.rayleigh_length*1e6:.2f} μm")
print(f"  Airy radius: {system.airy_radius*1e9:.0f} nm")

print(f"\nPulsed Beam:")
print(f"  Pulse width: {tau*1e15:.0f} fs")
print(f"  Spectral bandwidth: {pulsed_beam.delta_nu*1e-12:.1f} THz")
print(f"  Bandwidth (wavelength): {(pulsed_beam.lambda_max - pulsed_beam.lambda_min)*1e9:.0f} nm")

In [ ]:
# Create coordinate grids for contour plots
# Use ranges similar to the original paper figures
r_range = 20e-6  # ±20 μm radial range
z_range = 60e-6  # ±60 μm axial range  
nr = 101         # Radial points
nz = 121         # Axial points

# Create coordinate grids
r_coords, z_coords, R, Z = create_coordinate_grids(
    system, r_range=r_range, z_range=z_range, nr=nr, nz=nz
)

print(f"Coordinate ranges:")
print(f"  r: ±{r_range*1e6:.0f} μm ({nr} points)")
print(f"  z: ±{z_range*1e6:.0f} μm ({nz} points)")
print(f"  Total grid points: {nr*nz:,}")

In [ ]:
# Calculate CW intensity distribution
print("Calculating CW intensity distribution...")

# Flatten coordinates for calculation
r_flat = R.flatten()
z_flat = Z.flatten()

# Calculate electric field components using Richards-Wolf vector diffraction
Ex_cw, Ey_cw, Ez_cw = richards_wolf_vector_diffraction(system, r_flat, z_flat)

# Calculate intensity |E|² = |Ex|² + |Ey|² + |Ez|²
I_cw_flat = np.abs(Ex_cw)**2 + np.abs(Ey_cw)**2 + np.abs(Ez_cw)**2

# Reshape back to grid
I_cw = I_cw_flat.reshape(R.shape)

# Normalize to peak value of 100 (as in original figure)
I_cw_normalized = 100 * I_cw / np.max(I_cw)

print(f"CW calculation complete. Peak intensity: {np.max(I_cw_normalized):.1f}")

In [ ]:
# Calculate pulsed beam intensity distribution
print("Calculating 1-fs pulsed beam intensity distribution...")

# Use spectral integration approach (as in Romallosa paper)
# The pulsed intensity is calculated by integrating over the spectral bandwidth
I_pulsed_flat = pulsed_richards_wolf_diffraction(
    system, pulsed_beam, r_flat, z_flat, 
    n_spectral=201  # Use 201 frequency components as in the paper
)

# Reshape back to grid
I_pulsed = I_pulsed_flat.reshape(R.shape)

# Normalize to peak value of 100
I_pulsed_normalized = 100 * I_pulsed / np.max(I_pulsed)

print(f"Pulsed beam calculation complete. Peak intensity: {np.max(I_pulsed_normalized):.1f}")

In [ ]:
# Create contour plots (Figure 1 reproduction)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Convert coordinates to micrometers for plotting
r_um = r_coords * 1e6
z_um = z_coords * 1e6

# Define contour levels (matching the original paper style)
levels = [0, 1, 2, 4, 8, 16, 24, 32, 40, 50, 60, 70, 80, 90, 95, 100]

# Plot (a) CW beam
cs1 = ax1.contour(r_um, z_um, I_cw_normalized, levels=levels, colors='black', linewidths=0.8)
ax1.clabel(cs1, inline=True, fontsize=8, fmt='%g')
ax1.set_xlabel('r (μm)')
ax1.set_ylabel('z (μm)')
ax1.set_title('(a) CW beam')
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')
ax1.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax1.axvline(0, color='gray', linewidth=0.5, alpha=0.5)

# Plot (b) 1-fs pulsed beam  
cs2 = ax2.contour(r_um, z_um, I_pulsed_normalized, levels=levels, colors='black', linewidths=0.8)
ax2.clabel(cs2, inline=True, fontsize=8, fmt='%g')
ax2.set_xlabel('r (μm)')
ax2.set_ylabel('z (μm)')
ax2.set_title('(b) 1-fs optical pulse')
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')
ax2.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax2.axvline(0, color='gray', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.suptitle('Figure 1: Contour plots of time-integrated focused intensity distributions\n' +
             '(peak value = 100, minimum = 0) near the geometrical focus\n' +
             f'XNA = {NA}, λc = {wavelength*1e9:.0f} nm, n = {n}', 
             y=1.02, fontsize=14)
plt.show()

In [ ]:
# Additional analysis: Compare transverse and axial profiles
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# Transverse profiles at z=0
z_center_idx = nz // 2
I_cw_trans = I_cw_normalized[z_center_idx, :]
I_pulsed_trans = I_pulsed_normalized[z_center_idx, :]

ax1.plot(r_um, I_cw_trans, 'b-', linewidth=2, label='CW')
ax1.plot(r_um, I_pulsed_trans, 'r--', linewidth=2, label='1-fs pulsed')
ax1.set_xlabel('r (μm)')
ax1.set_ylabel('Normalized intensity')
ax1.set_title('(a) Transverse intensity profiles (z = 0)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-6, 6)

# Axial profiles at r=0
r_center_idx = nr // 2
I_cw_axial = I_cw_normalized[:, r_center_idx]
I_pulsed_axial = I_pulsed_normalized[:, r_center_idx]

ax2.plot(z_um, I_cw_axial, 'b-', linewidth=2, label='CW')
ax2.plot(z_um, I_pulsed_axial, 'r--', linewidth=2, label='1-fs pulsed')
ax2.set_xlabel('z (μm)')
ax2.set_ylabel('Normalized intensity')
ax2.set_title('(b) Axial intensity profiles (r = 0)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-10, 10)

# Log scale plots for better visualization of sidelobes
ax3.semilogy(r_um, np.maximum(I_cw_trans, 0.01), 'b-', linewidth=2, label='CW')
ax3.semilogy(r_um, np.maximum(I_pulsed_trans, 0.01), 'r--', linewidth=2, label='1-fs pulsed')
ax3.set_xlabel('r (μm)')
ax3.set_ylabel('Normalized intensity (log)')
ax3.set_title('(c) Transverse profiles (log scale)')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xlim(-6, 6)

ax4.semilogy(z_um, np.maximum(I_cw_axial, 0.01), 'b-', linewidth=2, label='CW')
ax4.semilogy(z_um, np.maximum(I_pulsed_axial, 0.01), 'r--', linewidth=2, label='1-fs pulsed')
ax4.set_xlabel('z (μm)')
ax4.set_ylabel('Normalized intensity (log)')
ax4.set_title('(d) Axial profiles (log scale)')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-10, 10)

plt.tight_layout()
plt.show()

In [ ]:
# Quantitative analysis
print("\n=== QUANTITATIVE ANALYSIS ===")
print(f"\nSystem parameters:")
print(f"  λc = {wavelength*1e9:.0f} nm")
print(f"  XNA = {NA}")
print(f"  n = {n}")
print(f"  τ = {tau*1e15:.0f} fs")

# Find FWHM of transverse profiles
def find_fwhm(x, y):
    """Find full width at half maximum."""
    half_max = np.max(y) / 2
    indices = np.where(y >= half_max)[0]
    if len(indices) > 0:
        return x[indices[-1]] - x[indices[0]]
    return 0

fwhm_cw_trans = find_fwhm(r_um, I_cw_trans)
fwhm_pulsed_trans = find_fwhm(r_um, I_pulsed_trans)
fwhm_cw_axial = find_fwhm(z_um, I_cw_axial)
fwhm_pulsed_axial = find_fwhm(z_um, I_pulsed_axial)

print(f"\nFWHM analysis:")
print(f"  Transverse FWHM (CW): {fwhm_cw_trans:.2f} μm")
print(f"  Transverse FWHM (pulsed): {fwhm_pulsed_trans:.2f} μm")
print(f"  Transverse broadening: {fwhm_pulsed_trans/fwhm_cw_trans:.3f}x")
print(f"  Axial FWHM (CW): {fwhm_cw_axial:.2f} μm")
print(f"  Axial FWHM (pulsed): {fwhm_pulsed_axial:.2f} μm") 
print(f"  Axial broadening: {fwhm_pulsed_axial/fwhm_cw_axial:.3f}x")

# Analyze zero crossings
print(f"\nZero crossings analysis:")
# Find minima in transverse profiles
cw_trans_minima = np.where((I_cw_trans[1:-1] < I_cw_trans[:-2]) & 
                          (I_cw_trans[1:-1] < I_cw_trans[2:]))[0] + 1
pulsed_trans_minima = np.where((I_pulsed_trans[1:-1] < I_pulsed_trans[:-2]) & 
                              (I_pulsed_trans[1:-1] < I_pulsed_trans[2:]))[0] + 1

print(f"  CW transverse minima count: {len(cw_trans_minima)}")
print(f"  Pulsed transverse minima count: {len(pulsed_trans_minima)}")
print(f"  CW minimum values: {I_cw_trans[cw_trans_minima] if len(cw_trans_minima) > 0 else 'None'}")
print(f"  Pulsed minimum values: {I_pulsed_trans[pulsed_trans_minima] if len(pulsed_trans_minima) > 0 else 'None'}")

print(f"\n=== KEY FINDINGS ===")
print(f"1. The 1-fs pulsed beam shows broader intensity distribution compared to CW")
print(f"2. Transverse profile broadening: {fwhm_pulsed_trans/fwhm_cw_trans:.1%}")
print(f"3. Axial profile broadening: {fwhm_pulsed_axial/fwhm_cw_axial:.1%}")
if len(pulsed_trans_minima) < len(cw_trans_minima):
    print(f"4. Pulsed beam has fewer/higher minima, reducing contrast")
else:
    print(f"4. Similar sidelobe structure maintained")

## Summary

This notebook reproduces **Figure 1** from Romallosa et al. (2003), showing:

### Key Results:
1. **CW beam** produces sharp intensity distribution with clear zeros/minima
2. **1-fs pulsed beam** shows:
   - Broader central spot
   - Higher minimum values (fewer true zeros)
   - Reduced contrast due to spectral broadening

### Physical Interpretation:
- The spectral bandwidth of a 1-fs pulse spans from ~483 nm to 1.67 μm
- Different spectral components focus at slightly different positions
- Incoherent summation leads to "filling in" of zeros
- Results in degraded spatial resolution and contrast

### Applications:
- Critical for ultrafast microscopy design
- Affects contrast and resolution in few-cycle pulse imaging
- Important for optimizing pulse duration vs spatial resolution trade-offs